# 04_export_and_validate

현재 기본값은 `CAMPAIGN="ucface_review"`, `EXECUTE=False`입니다. 기존 학습 기록은 보존합니다.

우선 목표는 UCFace Table I의 일반 ArcFace/CosFace/MagFace/AdaFace baseline 비교입니다. `continuous_margin`은 기본 목록에서 제외했습니다. 1024D는 본문 IV-A.3과 부록의 Table I 설명에 따릅니다. 저자 이메일의 512D 기억은 부록 Table XVII와 구분해 기록합니다.

아래 12개 목록은 **기존 연구 분할·정렬 조건을 보존한 후보**이며, UCFace의 동일 모델·동일 성능이 검증된 목록이 아닙니다. SurvFace 4,255명과 논문 5,319명의 차이, SCFace 인물 목록 및 정렬 세부 조건을 먼저 확인합니다. 검토 모드에서 EXECUTE=True로 바꾸어도 학습/export는 시작되지 않습니다.

`selected`는 명시적으로 기존 연구 조건을 이어가는 경로입니다. 기존 출력은 이전 실행의 기록이므로 커널을 재시작하고 Run All로 현재 계획을 확인하세요.


## 1. 설정
00에서 저장한 설정을 이어받으며 HEADS·SEEDS·FINETUNE_DATASETS와 부모·backbone·경로는 03의 완료 모델과 같아야 합니다. 04는 학습하지 않고 완료된 backbone을 export합니다. 새 캠페인은 앞 단계를 먼저 완료해야 합니다. CosFace metadata의 head=cosface, source_config.head.type=AM_Softmax입니다. SETTINGS_PATH는 00과 같아야 합니다.

선택지·범위가 적힌 변수만 필요한 조건으로 조정합니다. 공통 변경은 00과 같은 SETTINGS_PATH를 사용하는 후속 단계에 일관되게 반영합니다.


In [1]:
# 목적: 03의 완료 모델과 일치하는 조건을 읽고 export 대상과 실행 여부를 지정합니다.

from pathlib import Path
import sys, json, os
BASE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fine-tune/modules").is_dir())  # 저장소 위치 자동 탐색값입니다. 직접 변경하지 않습니다.
DATASET_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets" if os.name == "nt" else "/data/face_datasets"))  # 공유 데이터 루트; 환경변수 FACE_DATASET_ROOT 또는 이 Path를 실제 경로로 지정합니다.
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models" if os.name == "nt" else "/data/models"))  # 공유 모델 루트; 환경변수 FACE_MODEL_ROOT 또는 이 Path를 실제 경로로 지정합니다.
SETTINGS_PATH = DATASET_ROOT / ".state/facex/shared_v1_boundary/notebook_settings_mobilefacenet.json"  # 00에서 저장하고 01~04에서 함께 읽을 JSON입니다. backbone/실험별로 파일을 구분합니다.
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
from modules.finetune_protocol import default_profiles
CAMPAIGN = "selected"  # ucface_review=재현 검토 / selected=기존 연구 조건 / baseline=과거 resize 캠페인
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)

# selected는 기존 연구 조건이며 UCFace 동일 재현을 뜻하지 않습니다. 기존 셀 출력은 이전 실행 기록입니다.
SELECTED_ROOT = MODEL_ROOT / "facex/selected/ucface_recipes_v1"
SELECTION_FILE = SELECTED_ROOT / "plans/selection_v2_local_protocol.json"  # 03과 04에서 같은 파일. 목록 변경 시 v2 등 새 파일명.

# 검증 완료 후 원본/전처리 이미지와 목록을 임의로 변경하지 않는다는 전제입니다.
DATA_INTEGRITY_MODE = "immutable_verified_reuse"  # 최초 전체 검증 후 JSON 기록 재사용; "content_verified_reuse"=매번 hash, "full"=매번 hash+decode.
FORCE_DATA_REVERIFY = False  # True=이번 실행에서 전체 이미지 hash+decode 재검증. 성공 후 False로 되돌리세요.
SEEDS = list(saved["project"]["seed_list"])  # 중복 없는 정수 목록(0 <= seed < 2**32); 일부 seed만 선택 가능하며 후속 단계도 동일하게 맞춥니다.
HEADS = list(saved["campaign"]["heads"])  # "arcface", "adaface", "magface", "cosface" 중 중복 없이 선택; 02 부모는 ArcFace로 고정입니다.
PARENT_SEED = saved["campaign"]["parent_seed"]  # 공통 부모 seed(0 <= seed < 2**32 정수); 후속 head/seed가 공유하므로 02~04에서 유지합니다.
PRETRAIN_EPOCHS = saved["campaign"]["pretrain_epochs"]  # 부모 학습 epoch 수(양의 정수, 기본 40); 재사용 부모의 완료 epoch와 일치해야 합니다.
PRETRAIN_LR = saved["campaign"]["pretrain_lr"]  # parent의 기존 LR을 별도 유지합니다. downstream 논문 LR과 구분합니다.
PARENT_CHECKPOINT = saved["campaign"]["parent_checkpoint"]  # None=02의 고정 경로; 다른 완료 부모를 쓰려면 정확한 경로와 PARENT_SHA256를 함께 지정합니다.
PARENT_SHA256 = saved["campaign"]["parent_sha256"]  # None 또는 지정한 부모 파일의 64자리 SHA-256; 경로와 함께 지정하며 학습 조건도 검증합니다.
FINETUNE_DATASETS = list(saved["campaign"]["finetune_datasets"])  # "survface", "tinyface", "scface" 중 선택합니다. LFW/RFW는 평가 전용입니다.
FINETUNE_PROFILES = saved["finetune_profiles"]  # 외부 protocol의 분할·경로·hash입니다. 분할 인물/수량/hash를 여기서 임의로 고치지 않습니다.
AUDIT_WORKERS = 4  # 로컬 release 검사 worker 수(1 이상 정수); 현재 portable 공유 protocol에서는 사용하지 않습니다.
# 원본 데이터·공유 전처리·부모·과거 실패 기록은 이 보존 옵션의 삭제 대상이 아닙니다.
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]  # 03의 보존 설정을 이어받습니다. 04의 export는 학습 checkpoint를 정리하지 않습니다.
NUM_WORKERS = saved["training"]["num_workers"]  # DataLoader worker 수(0~12 정수); 0=현재 프로세스에서 로딩, 큰 값은 CPU/RAM을 더 사용합니다.
PREFETCH_FACTOR = saved["training"]["prefetch_factor"]  # worker당 미리 읽을 batch 수(1 이상 정수); NUM_WORKERS>0일 때 적용합니다.
PERSISTENT_WORKERS = saved["training"]["persistent_workers"]  # True=epoch 사이 worker 유지, False=다시 생성; NUM_WORKERS=0이면 적용되지 않습니다.
TORCH_THREADS = saved["training"]["torch_threads"]  # PyTorch CPU 연산 thread 수(1 이상 정수); GPU 수나 DataLoader worker 수와 다릅니다.
LOSS_SYNC_BATCHES = saved["training"].get("loss_sync_batches", 20)  # loss 집계값을 CPU로 가져오는 batch 간격(1 이상 정수); 매 batch 학습 업데이트는 유지합니다.
RECOVERY_MINUTES = saved["training"]["recovery_minutes"]  # 재개용 checkpoint 저장 간격(분, 0보다 큰 값); 짧을수록 복구 간격과 디스크 쓰기가 촘촘해집니다.
DEVICE = saved["project"]["device"]  # 저장된 학습 장치 설정입니다. 이 값을 읽는다고 04에서 GPU 학습이 시작되지는 않습니다.
DATASET = dict(saved["dataset"])  # VGGFace2 경로·입력 설정입니다. 112×112와 pixel_mean=127.5/pixel_std=128.0 조건을 유지합니다.
BACKBONE = dict(saved["backbone"])  # 00에서 MobileFaceNet/Resnet IR-50 선택; 1024D·dropout 0.6·out_h/out_w=7 조건을 유지합니다.
# 정식 fine-tune: optimizer=adam, lr=MobileFaceNet 0.001 / ResNet-50·100 0.0001, epoches=40, milestones=[10,20,30], lr_decay_ratio=0.1 고정.
# baseline batch_size=64는 MobileFaceNet·ResNet-50/100 공통; save_freq는 1 이상 epoch, drop_last=False 유지.
TRAINING = dict(saved["training"])  # 학습/저장 설정 사전입니다. 위 고정 학습 조건과 조절 가능한 저장 간격을 구분합니다.
HEAD_PARAMETERS = saved["heads"]  # head별 margin/scale 기준값입니다. 정식 fine-tune은 이를 검증하므로 head 선택은 HEADS에서 합니다.
PARENT_HEAD = dict(saved["parent_head"])  # 부모는 ArcFace, margin_arc=0.35/margin_am=0/scale=32로 고정합니다.
# flip_probability/affine_probability: 0~1, rotation_degrees: 0 이상, scale_range=[low,high]: 0<low<=high.
# border_mode="reflect"/"constant", center="image_center" 고정; 현재 수치를 UCFace 명시값으로 간주하지 않습니다.
AUGMENTATION = dict(saved["augmentation"])  # 증강 수치 변경은 별도 학습 조건입니다. 새 namespace와 부모 학습으로 구분합니다.
ALIGNMENT = dict(saved["alignment"])  # VGGFace2=official_landmarks, 저해상도=resize_only; 보간법은 bicubic/bilinear 중 선택합니다.
PATHS = dict(saved["paths"])  # checkpoint/export/log/report/cache 경로; 00에서 namespace별로 정하고 후속 단계가 이어받습니다.
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]  # True=동일 조건의 검증된 완료 결과 재사용; False=완료 경로가 있으면 중단하며 덮어쓰지 않습니다.
EXECUTE = True  # False=계획 확인, True=분할·완료 모델 검증과 export 수행; 새 학습은 실행하지 않습니다.
STAGE = "export"  # 이 노트북의 고정 단계입니다. 다른 단계는 해당 번호의 노트북을 사용합니다.
# 위 변수들을 실제 모듈 설정으로 묶어 검증합니다. 아래 전달부는 직접 편집할 필요가 없습니다.
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
 "data_integrity":dict(mode=DATA_INTEGRITY_MODE,force_reverify=FORCE_DATA_REVERIFY),
 "project":dict(seed=SEEDS[0],seed_list=SEEDS,keep_raw_results=KEEP_RAW_RESULTS,device=DEVICE),
 "campaign":dict(heads=HEADS,reuse_completed=REUSE_COMPLETED,parent_seed=PARENT_SEED,
    pretrain_epochs=PRETRAIN_EPOCHS,pretrain_lr=PRETRAIN_LR,parent_checkpoint=PARENT_CHECKPOINT,parent_sha256=PARENT_SHA256,
    finetune_datasets=FINETUNE_DATASETS),
 "finetune_profiles":FINETUNE_PROFILES,"dataset":DATASET,"backbone":BACKBONE,"heads":HEAD_PARAMETERS,
 "training":{**TRAINING,"num_workers":NUM_WORKERS,"prefetch_factor":PREFETCH_FACTOR,
    "persistent_workers":PERSISTENT_WORKERS,"torch_threads":TORCH_THREADS,"recovery_minutes":RECOVERY_MINUTES,"loss_sync_batches":LOSS_SYNC_BATCHES},
 "paths":PATHS,"parent_head":PARENT_HEAD,"augmentation":AUGMENTATION,"alignment":ALIGNMENT})
if CAMPAIGN in ("selected", "ucface_review"):
    from modules.selected_campaign import load_selection, inspect_selection
    selection = load_selection(SELECTION_FILE)  # 03에서 확정한 목록을 그대로 읽습니다.
elif CAMPAIGN != "baseline":
    raise ValueError(f"Unknown campaign: {CAMPAIGN}")

from modules.ucface_reference import reproduction_review, require_reproduction_ready


## 2. 대상

In [2]:
# 목적: export할 데이터셋·head·seed와 필요한 완료 epoch를 확인합니다.

plan = inspect_selection(selection, force_reverify=FORCE_DATA_REVERIFY) if CAMPAIGN in ("selected", "ucface_review") else ([{"dataset": d, "stage": "export"} for d in ("scface", "tinyface")]
        if CAMPAIGN == "aligned_controls" else plan_matrix(settings, STAGE))
for row in plan:
    print(row)
print("TARGET_EPOCH:",TRAINING["epoches"],"예정:",len(plan))
if CAMPAIGN in ("selected", "ucface_review"):
    print("UCFace 재현 검토:", reproduction_review(selection))


{'run_id': 'scface_adaface_two_eye_alignment_s8972', 'dataset': 'scface', 'head': 'adaface', 'recipe': 'two_eye_alignment', 'seed': 8972, 'status': 'reuse_completed', 'checkpoint': 'F:\\models\\facex\\selected\\ucface_recipes_v1\\checkpoint\\scface_adaface_two_eye_alignment_s8972\\finetune_scface_adaface_seed8972\\finetune_mobilefacenet_1024d_adaface_epoch_40.pt', 'target': 'F:\\models\\facex\\selected\\ucface_recipes_v1\\checkpoint\\scface_adaface_two_eye_alignment_s8972\\finetune_scface_adaface_seed8972\\finetune_mobilefacenet_1024d_adaface_epoch_40.pt'}
{'run_id': 'scface_arcface_two_eye_alignment_s8972', 'dataset': 'scface', 'head': 'arcface', 'recipe': 'two_eye_alignment', 'seed': 8972, 'status': 'reuse_completed', 'checkpoint': 'F:\\models\\facex\\diagnostics\\ucface_alignment_arcface_s8972_20261112_v1\\checkpoints\\finetune_scface_arcface_seed8972\\finetune_mobilefacenet_1024d_arcface_epoch_40.pt', 'target': 'F:\\models\\facex\\selected\\ucface_recipes_v1\\checkpoint\\scface_arc

## 3. 출처 기준

In [3]:
if EXECUTE and CAMPAIGN == "ucface_review":
    require_reproduction_ready(selection)

# 목적: 공통 부모 정보를 확인하고, 실행 시 고정 분할을 검증해 export용 설정을 저장합니다.

print("공통 부모:", PARENT_CHECKPOINT or "02의 PARENT_SEED 고정 경로", "parent seed:", PARENT_SEED)
print("시작 전 00 설정 저장과 01 데이터 준비를 확인하세요.")
from modules.finetune_protocol import prepare_selected_protocols
if EXECUTE and CAMPAIGN == "baseline":
    for protocol in prepare_selected_protocols(settings, audit_workers=AUDIT_WORKERS):
        print(protocol)
    save_settings(SETTINGS_PATH, settings)
else:
    print("계획 모드: 공식 split 및 전체 이미지 검증/목록 생성 미실행")

공통 부모: 02의 PARENT_SEED 고정 경로 parent seed: 8972
시작 전 00 설정 저장과 01 데이터 준비를 확인하세요.
계획 모드: 공식 split 및 전체 이미지 검증/목록 생성 미실행


## 4. 별도 프로세스로 export 실행

완료 epoch·설정·분할·부모 hash를 검증한 뒤 backbone과 metadata를 게시합니다. 학습은 수행하지 않으며 중단 시 결과 상태를 확인한 뒤 같은 조건으로 다시 실행합니다.


In [4]:
if EXECUTE and CAMPAIGN == "ucface_review":
    require_reproduction_ready(selection)

# 목적: 실행 시 완료 checkpoint를 검증해 backbone 가중치와 대응 metadata를 내보냅니다.

from modules.process_runner import launch_campaign
results = []
if EXECUTE and CAMPAIGN == "selected":
    results = launch_campaign(BASE_DIR, settings, STAGE, selection=selection, force_reverify=FORCE_DATA_REVERIFY)
elif EXECUTE and CAMPAIGN == "aligned_controls":
    results = [export_alignment_control(settings, row["dataset"], CONTROL_SOURCE, CONTROL_EXPORT) for row in plan]
elif EXECUTE:
    results=launch_campaign(BASE_DIR,settings,STAGE)
else:
    print("계획 또는 benchmark 모드: 정식 작업 미실행")

Child PID 42412; log: F:\face_datasets\.cache\facex\launches\shared_v1_boundary_mobilefacenet\383a97d108114811b293f85e3e585f59\console.log; stop request: F:\face_datasets\.cache\facex\launches\shared_v1_boundary_mobilefacenet\383a97d108114811b293f85e3e585f59\stop.request
2026-11-15 19:10:10,807 INFO Built FaceRecognitionModel: Backbone=MobileFaceNet(dim=1024, dropout=0.6), Head=AdaFace(num_class=50)
2026-11-15 19:10:11,274 INFO Built FaceRecognitionModel: Backbone=MobileFaceNet(dim=1024, dropout=0.6), Head=ArcFace(num_class=50)
2026-11-15 19:10:11,750 INFO Built FaceRecognitionModel: Backbone=MobileFaceNet(dim=1024, dropout=0.6), Head=CosFace(num_class=50)
2026-11-15 19:10:12,197 INFO Built FaceRecognitionModel: Backbone=MobileFaceNet(dim=1024, dropout=0.6), Head=MagFace(num_class=50)
2026-11-15 19:10:12,875 INFO Built FaceRecognitionModel: Backbone=MobileFaceNet(dim=1024, dropout=0.6), Head=AdaFace(num_class=2570)
2026-11-15 19:10:13,613 INFO Built FaceRecognitionModel: Backbone=Mobil

## 5. 결과

In [5]:
# 목적: export 완료 여부와 결과 경로를 확인하고 ronbun 평가는 별도 단계임을 표시합니다.

for result in results:
    print(result)
if CAMPAIGN in ("selected", "ucface_review") and not EXECUTE:
    print("계획 확인:", {status: sum(r["status"] == status for r in plan) for status in ("reuse_completed", "train_new", "resume", "blocked")})
    print("이번 실행에서 학습/export하지 않았습니다.")
else:
    print("완료 export:",sum(r.get("completed") is True for r in results),"/",len(plan))
if EXECUTE:
    print("strict load/shape 확인. 실제 ronbun·Continuous FIQA 평가는 미실행.")
print("해석 안내:",BASE_DIR/"fine-tune/README.md")


{'run_id': 'scface_adaface_two_eye_alignment_s8972', 'dataset': 'scface', 'head': 'adaface', 'recipe': 'two_eye_alignment', 'seed': 8972, 'completed': True, 'export': 'F:\\models\\facex\\selected\\ucface_recipes_v1\\exports\\scface_adaface_two_eye_alignment_s8972\\scface_adaface_two_eye_alignment_s8972.pt', 'sha256': '8e44ba03d0434ff51187e9b5dbf9521d5dbf7c45350f43775c5a10483de1a339', 'source_checkpoint': 'F:\\models\\facex\\selected\\ucface_recipes_v1\\checkpoint\\scface_adaface_two_eye_alignment_s8972\\finetune_scface_adaface_seed8972\\finetune_mobilefacenet_1024d_adaface_epoch_40.pt', 'source_sha256': '5998e85019adfdebde8c14a0f00652e394174308bbc3b8520a258c1627c4a956', 'evaluation_datasets': ['scface'], 'training_dataset': 'scface'}
{'run_id': 'scface_arcface_two_eye_alignment_s8972', 'dataset': 'scface', 'head': 'arcface', 'recipe': 'two_eye_alignment', 'seed': 8972, 'completed': True, 'export': 'F:\\models\\facex\\exports\\ucface_alignment_arcface_s8972_20261112_v1\\scface_arcface_a